In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import time 



df = pd.read_csv(r"data\raw\Telco_customer.csv")
df.shape

(7043, 33)

In [2]:
df = df.drop_duplicates()

In [3]:
df.drop(columns=['Churn Reason' , 'Churn Label' , 'Count' , 'Lat Long' , 'CustomerID' , 'Zip Code' , 'Country', 'State', 'City', 'Latitude', 'Longitude' , 'Churn Score' , 'CLTV'] , inplace=True)


In [4]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

# 2. Fill those new NaN values with 0 (since new customers have 0 charges)
df['Total Charges'] = df['Total Charges'].fillna(0)

# 3. Check the data type to make sure it is now a float
print(df['Total Charges'].dtype)

float64


In [5]:
df['Gender'] = df['Gender'].map({'Male': 1 , 'Female': 0})
df['Senior Citizen'] = df['Senior Citizen'].map({'Yes': 1 , 'No': 0})
df['Partner'] = df['Partner'].map({'Yes': 1 , 'No': 0})
df['Dependents'] = df['Dependents'].map({'Yes': 1 , 'No': 0})
df['Phone Service'] = df['Phone Service'].map({'Yes': 1 , 'No': 0})
df['Paperless Billing'] = df['Paperless Billing'].map({'Yes': 1 , 'No': 0})

In [6]:
from sklearn.preprocessing import StandardScaler , OneHotEncoder

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split , cross_val_score
from sklearn.metrics import accuracy_score , confusion_matrix , classification_report , ConfusionMatrixDisplay , RocCurveDisplay , PrecisionRecallDisplay , roc_auc_score , precision_score , recall_score , f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier , GradientBoostingClassifier 
from xgboost import XGBClassifier
from catboost import CatBoostClassifier 
from lightgbm import LGBMClassifier 
from imblearn.over_sampling import SMOTE 
from imblearn.pipeline import Pipeline



In [7]:
x = df.drop(columns=['Churn Value'])
y = df['Churn Value']


In [8]:
X_dev, X_prod, y_dev, y_prod = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X_dev,
    y_dev,
    test_size=0.20,
    random_state=42,
    stratify=y_dev
)

In [10]:
categorical = ['Multiple Lines' , 'Internet Service' , 'Online Security' , 'Online Backup' , 'Device Protection' , 'Tech Support' , 'Streaming TV' , 'Streaming Movies' , 'Contract' , 'Payment Method']
num_cols = ['Monthly Charges' , 'Total Charges' , 'Tenure Months']
preprocessor = ColumnTransformer(
    transformers=[
    ("onehot" , OneHotEncoder(drop='first') , categorical) ,
    ("standard" , StandardScaler() , num_cols)
] , remainder='passthrough')

In [11]:
pipeline = Pipeline([
    ("preprocessor" , preprocessor) ,
    ("smote" , SMOTE(sampling_strategy=0.6 , random_state=42)) ,
    ("cat" , CatBoostClassifier(verbose=0 , random_state=42 , max_depth=5 , learning_rate=0.03 , n_estimators=300))
])

pipeline.fit(X_train , y_train)
pred = pipeline.predict(X_test)
acc = accuracy_score(y_test , pred)
clr = classification_report(y_test , pred)
roc = roc_auc_score(y_test , pipeline.predict_proba(X_test)[: , 1])
cv = cross_val_score(
    pipeline ,
    X_train ,
    y_train ,
    cv = 5 ,
    scoring='f1'
).mean()
print("accu" , acc)
print(clr)
print("roc" , roc)
print("cv" , cv)

accu 0.8260869565217391
              precision    recall  f1-score   support

           0       0.87      0.89      0.88       828
           1       0.69      0.64      0.66       299

    accuracy                           0.83      1127
   macro avg       0.78      0.77      0.77      1127
weighted avg       0.82      0.83      0.82      1127

roc 0.8668892281841242
cv 0.6341466087294808


In [ ]:
import joblib 
joblib.dump(pipeline , r"models\v1\produc_model.pkl")

['C:\\Users\\ah266\\OneDrive\\Documents\\production-ml-reliability\\models\\v1\\produc_model.pkl']

In [18]:
n_batches = 5

batch_size = len(X_prod) // n_batches

X_batches = []
y_batches = []

for i in range(n_batches):
    
    start = i * batch_size
    
    if i == n_batches - 1:
        end = len(X_prod)
    else:
        end = (i + 1) * batch_size
    
    X_batches.append(X_prod.iloc[start:end])
    y_batches.append(y_prod.iloc[start:end])

In [19]:
X_batch = X_batches[0]
y_actual = y_batches[0]

In [20]:
start_time = time.perf_counter()

predictions = pipeline.predict(X_batch)

end_time = time.perf_counter()

latency = end_time - start_time

print("Prediction latency:", latency)

Prediction latency: 0.8590193999698386


In [23]:
probabilities = pipeline.predict_proba(X_batch)[:, 1]

accuracy = accuracy_score(
    y_actual,
    predictions
)

precision = precision_score(
    y_actual,
    predictions
)

recall = recall_score(
    y_actual,
    predictions
)

f1 = f1_score(
    y_actual,
    predictions
)

roc_auc = roc_auc_score(
    y_actual,
    probabilities
)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("ROC-AUC:", roc_auc)

Accuracy: 0.800711743772242
Precision: 0.6447368421052632
Recall: 0.6282051282051282
F1: 0.6363636363636364
ROC-AUC: 0.8640899330554502


In [24]:
prediction_log = pd.DataFrame({
    "model_version": "v1",
    "prediction": predictions,
    "prediction_probability": probabilities,
    "actual_label": y_actual.values
})

prediction_log.head()

,model_version,prediction,prediction_probability,actual_label
0,v1,0,0.065950,0
1,v1,1,0.790079,0
2,v1,0,0.112348,0
3,v1,0,0.413663,0
4,v1,0,0.040054,0


In [25]:
prediction_log["batch_id"] = "batch_001"
prediction_log["timestamp"] = pd.Timestamp.now()
prediction_log.head()

,model_version,prediction,prediction_probability,actual_label,batch_id,timestamp
0,v1,0,0.065950,0,batch_001,2026-08-28 22:19:32.067914
1,v1,1,0.790079,0,batch_001,2026-08-28 22:19:32.067914
2,v1,0,0.112348,0,batch_001,2026-08-28 22:19:32.067914
3,v1,0,0.413663,0,batch_001,2026-08-28 22:19:32.067914
4,v1,0,0.040054,0,batch_001,2026-08-28 22:19:32.067914


In [26]:
prediction_log["batch_latency_seconds"] = latency

In [27]:
def process_batch(X_batch, y_actual, batch_id, model, model_version):

    start_time = time.perf_counter()

    predictions = model.predict(X_batch)
    probabilities = model.predict_proba(X_batch)[:, 1]

    end_time = time.perf_counter()

    latency = end_time - start_time

    metrics = {
        "batch_id": batch_id,
        "model_version": model_version,
        "accuracy": accuracy_score(y_actual, predictions),
        "precision": precision_score(y_actual, predictions),
        "recall": recall_score(y_actual, predictions),
        "f1": f1_score(y_actual, predictions),
        "roc_auc": roc_auc_score(y_actual, probabilities),
        "latency_seconds": latency
    }

    logs = pd.DataFrame({
        "batch_id": batch_id,
        "model_version": model_version,
        "prediction": predictions,
        "prediction_probability": probabilities,
        "actual_label": y_actual.values,
        "timestamp": pd.Timestamp.now()
    })

    return logs, metrics

In [28]:
all_logs = []
all_metrics = []

for i, (X_batch, y_batch) in enumerate(
    zip(X_batches, y_batches),
    start=1
):

    logs, metrics = process_batch(
        X_batch,
        y_batch,
        f"batch_{i:03d}",
        pipeline,
        "v1"
    )

    all_logs.append(logs)
    all_metrics.append(metrics)

In [29]:
prediction_logs = pd.concat(
    all_logs,
    ignore_index=True
)

production_metrics = pd.DataFrame(
    all_metrics
)

In [30]:
production_metrics

,batch_id,model_version,accuracy,precision,recall,f1,roc_auc,latency_seconds
0,batch_001,v1,0.800712,0.644737,0.628205,0.636364,0.864090,0.173268
1,batch_002,v1,0.782918,0.573770,0.500000,0.534351,0.811510,0.037735
2,batch_003,v1,0.790036,0.620253,0.628205,0.624204,0.849312,0.038704
3,batch_004,v1,0.775801,0.530864,0.632353,0.577181,0.838822,0.038333
4,batch_005,v1,0.824561,0.702703,0.650000,0.675325,0.909878,0.049086


In [ ]:
prediction_logs.to_csv(
    r"data\production\prediction_logs.csv",
    index=False
)

production_metrics.to_csv(
    r"data\processed\production_metrics.csv",
    index=False
)

In [ ]:
X_train.to_csv(
    r"data\processed\reference_data.csv",
    index=False
)

In [ ]:
for i, X_batch in enumerate(X_batches, start=1):

    X_batch.to_csv(
        fr"data\production\batch_{i:03d}.csv",
        index=False
    )

In [ ]:
for i, y_batch in enumerate(y_batches, start=1):

    y_batch.to_csv(
        fr"data\production\labels_{i:03d}.csv",
        index=False
    )

In [ ]:
y_train.to_csv(
    r"data\processed\reference_labels.csv",
    index=False
)